In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

print("All libraries loaded")


All libraries loaded


In [6]:
cdc = pd.read_csv('data/cdc_overdose_deaths.csv')

In [7]:
cdc.shape

(18808, 10)

In [8]:
cdc.head()

,Notes,County,County Code,Year,Year Code,Deaths,Population,Crude Rate,Crude Rate Lower 95% Confidence Interval,Crude Rate Upper 95% Confidence Interval
0,NaN,"Autauga County, AL",1001.0,2018.0,2018.0,Suppressed,55601,Suppressed,Suppressed,Suppressed
1,NaN,"Autauga County, AL",1001.0,2019.0,2019.0,Suppressed,55869,Suppressed,Suppressed,Suppressed
2,NaN,"Autauga County, AL",1001.0,2020.0,2020.0,Suppressed,56145,Suppressed,Suppressed,Suppressed
3,NaN,"Autauga County, AL",1001.0,2021.0,2021.0,10,59095,16.9,8.1,31.1
4,NaN,"Autauga County, AL",1001.0,2022.0,2022.0,10,59759,16.7,8.0,30.8


In [9]:
print(cdc.columns.tolist())


['Notes', 'County', 'County Code', 'Year', 'Year Code', 'Deaths', 'Population', 'Crude Rate', 'Crude Rate Lower 95% Confidence Interval', 'Crude Rate Upper 95% Confidence Interval']


In [10]:
cdc.dtypes

Notes                                           str
County                                          str
County Code                                 float64
Year                                        float64
Year Code                                   float64
Deaths                                          str
Population                                      str
Crude Rate                                      str
Crude Rate Lower 95% Confidence Interval        str
Crude Rate Upper 95% Confidence Interval        str
dtype: object

In [11]:
print(cdc['Deaths'].value_counts().head(10))


Deaths
Suppressed    10354
10              494
11              428
12              332
13              298
14              268
15              257
17              252
16              239
18              223
Name: count, dtype: int64


In [12]:
print(cdc['Crude Rate'].value_counts().head(10))


Crude Rate
Suppressed    10354
20.6             37
30.0             34
23.0             31
20.9             31
20.7             31
19.7             31
25.4             31
22.3             30
26.7             30
Name: count, dtype: int64


In [13]:
print(cdc.shape)

(18808, 10)


In [14]:
# Drop columns we don't need
cdc = cdc.drop(columns=['Notes', 'Year Code', 
                         'Crude Rate Lower 95% Confidence Interval',
                         'Crude Rate Upper 95% Confidence Interval'])

In [15]:
# Rename to clean names
cdc.columns = ['county_name', 'county_fips', 'year', 'deaths', 'population', 'crude_rate']


In [16]:
# Flag suppressed rows BEFORE replacing — we want to keep this information
cdc['is_suppressed'] = cdc['deaths'] == 'Suppressed'


In [17]:
cdc

,county_name,county_fips,year,deaths,population,crude_rate,is_suppressed
0,"Autauga County, AL",1001.0,2018.0,Suppressed,55601,Suppressed,True
1,"Autauga County, AL",1001.0,2019.0,Suppressed,55869,Suppressed,True
2,"Autauga County, AL",1001.0,2020.0,Suppressed,56145,Suppressed,True
3,"Autauga County, AL",1001.0,2021.0,10,59095,16.9,False
4,"Autauga County, AL",1001.0,2022.0,10,59759,16.7,False
...,...,...,...,...,...,...,...
18803,NaN,NaN,NaN,NaN,NaN,NaN,False
18804,NaN,NaN,NaN,NaN,NaN,NaN,False
18805,NaN,NaN,NaN,NaN,NaN,NaN,False
18806,NaN,NaN,NaN,NaN,NaN,NaN,False


In [18]:
# Replace Suppressed with NaN so we can convert to numeric
cdc['deaths'] = cdc['deaths'].replace('Suppressed', np.nan)
cdc['crude_rate'] = cdc['crude_rate'].replace('Suppressed', np.nan)

In [20]:
# Convert all numeric columns from string to number
cdc['deaths'] = pd.to_numeric(cdc['deaths'])
cdc['crude_rate'] = pd.to_numeric(cdc['crude_rate'], errors='coerce')
cdc['population'] = pd.to_numeric(cdc['population'], errors='coerce')


In [24]:
cdc['crude_rate']

0         NaN
1         NaN
2         NaN
3        16.9
4        16.7
         ... 
18803     NaN
18804     NaN
18805     NaN
18806     NaN
18807     NaN
Name: crude_rate, Length: 18808, dtype: float64

In [25]:
# Fix FIPS — remove decimal, pad to 5 digits
cdc['county_fips'] = cdc['county_fips'].astype(str).str.split('.').str[0].str.zfill(5)


In [26]:
cdc = cdc.dropna(subset=['county_name'])

In [27]:
print(cdc.shape)


(18738, 7)


In [28]:

print(cdc.dtypes)



county_name          str
county_fips       object
year             float64
deaths           float64
population       float64
crude_rate       float64
is_suppressed       bool
dtype: object


In [29]:
print(cdc.head())


          county_name county_fips    year  deaths  population  crude_rate  \
0  Autauga County, AL       01001  2018.0     NaN     55601.0         NaN   
1  Autauga County, AL       01001  2019.0     NaN     55869.0         NaN   
2  Autauga County, AL       01001  2020.0     NaN     56145.0         NaN   
3  Autauga County, AL       01001  2021.0    10.0     59095.0        16.9   
4  Autauga County, AL       01001  2022.0    10.0     59759.0        16.7   

   is_suppressed  
0           True  
1           True  
2           True  
3          False  
4          False  


In [30]:
cdc['year'] = cdc['year'].astype(int)


In [31]:
print(f"Total counties: {cdc['county_fips'].nunique()}")
print(f"Years covered: {sorted(cdc['year'].unique())}")
print(f"Suppressed rows: {cdc['is_suppressed'].sum()}")
print(f"Real death counts: {cdc['is_suppressed'].value_counts()}")


Total counties: 3059
Years covered: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Suppressed rows: 10354
Real death counts: is_suppressed
True     10354
False     8384
Name: count, dtype: int64


In [32]:
cdc.to_csv('data/cdc_clean.csv', index=False)
print("CDC data saved")


CDC data saved


In [33]:
oeps = pd.read_csv('data/oeps_pharmacy_merged.csv', 
                   dtype={'CountyGEOID': str, 'GEOID': str})

In [34]:
print(oeps.shape)
print(oeps.columns.tolist())
print(oeps['summary_level'].value_counts())

(85187, 37)
['Unnamed: 0.1', 'Unnamed: 0', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'AFFGEOID', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'NAMELSADCO', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'HEROP_ID', 'minx', 'miny', 'maxx', 'maxy', 'BBOX', 'LABEL', 'geometry', 'summary_level', 'FIPS', 'origin', 'PharCntDr', 'PharTmDr', 'PharTmDr2', 'CountyGEOID', 'TotTracts', 'PharCtTmDr', 'PharCtTmDr2', 'PharTmDrP', 'PharTmDrP2', 'PharAvTmDr', 'PharAvTmDr2']
summary_level
140    85187
Name: count, dtype: int64


In [35]:
oeps

,Unnamed: 0.1,Unnamed: 0,STATEFP,COUNTYFP,TRACTCE,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,...,PharTmDr,PharTmDr2,CountyGEOID,TotTracts,PharCtTmDr,PharCtTmDr2,PharTmDrP,PharTmDrP2,PharAvTmDr,PharAvTmDr2
0,0,0,1,89,11021,1400000US01089011021,01089011021,110.21,Census Tract 110.21,AL,...,4.56,9.12,01089,95,93,91,97.89,95.79,4.09,8.17
1,1,1,1,95,31200,1400000US01095031200,01095031200,312.00,Census Tract 312,AL,...,0.00,0.00,01095,26,26,19,100.00,73.08,8.64,17.27
2,2,2,1,73,12401,1400000US01073012401,01073012401,124.01,Census Tract 124.01,AL,...,0.00,0.00,01073,189,189,184,100.00,97.35,3.41,6.82
3,3,3,1,73,3400,1400000US01073003400,01073003400,34.00,Census Tract 34,AL,...,0.00,0.00,01073,189,189,184,100.00,97.35,3.41,6.82
4,4,4,1,73,10402,1400000US01073010402,01073010402,104.02,Census Tract 104.02,AL,...,3.88,7.76,01073,189,189,184,100.00,97.35,3.41,6.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85182,85182,85182,78,30,960800,1400000US78030960800,78030960800,9608.00,Census Tract 9608,VI,...,NaN,NaN,78030,12,0,0,0.00,0.00,NaN,NaN
85183,85183,85183,78,30,961200,1400000US78030961200,78030961200,9612.00,Census Tract 9612,VI,...,NaN,NaN,78030,12,0,0,0.00,0.00,NaN,NaN
85184,85184,85184,78,10,970300,1400000US78010970300,78010970300,9703.00,Census Tract 9703,VI,...,NaN,NaN,78010,15,0,0,0.00,0.00,NaN,NaN
85185,85185,85185,78,30,960200,1400000US78030960200,78030960200,9602.00,Census Tract 9602,VI,...,NaN,NaN,78030,12,0,0,0.00,0.00,NaN,NaN


In [36]:
# Separate tract level data
tract = oeps[['GEOID', 'CountyGEOID', 'PharCntDr', 'PharTmDr', 'PharTmDr2']].copy()

# Get county level by dropping duplicate counties
county = oeps[['CountyGEOID', 'TotTracts', 'PharCtTmDr', 'PharTmDrP', 'PharAvTmDr']].drop_duplicates(subset=['CountyGEOID'])

# Fix FIPS padding
tract['GEOID'] = tract['GEOID'].astype(str).str.zfill(11)
tract['CountyGEOID'] = tract['CountyGEOID'].astype(str).str.zfill(5)
county['CountyGEOID'] = county['CountyGEOID'].astype(str).str.zfill(5)

print("Tract rows:", tract.shape)
print("County rows:", county.shape)
print(county.head())

Tract rows: (85187, 5)
County rows: (3234, 5)
  CountyGEOID  TotTracts  PharCtTmDr  PharTmDrP  PharAvTmDr
0       01089         95          93      97.89        4.09
1       01095         26          26     100.00        8.64
2       01073        189         189     100.00        3.41
8       01103         31          31     100.00        4.62
9       01125         59          59     100.00        5.46


In [37]:
worst = county.sort_values('PharTmDrP').head(10)
print(worst[['CountyGEOID', 'PharCtTmDr', 'PharTmDrP', 'PharAvTmDr']])

      CountyGEOID  PharCtTmDr  PharTmDrP  PharAvTmDr
85170       78020           0        0.0         NaN
68243       46085           0        0.0       41.44
4278        06091           0        0.0       45.28
68237       46041           0        0.0       83.81
68221       46117           0        0.0       30.48
68219       46071           0        0.0       67.92
68202       46137           0        0.0         NaN
68200       46123           0        0.0       82.04
68194       46075           0        0.0       66.73
63111       41069           0        0.0       74.21


In [38]:
tract.to_csv('data/oeps_tract_clean.csv', index=False)
county.to_csv('data/oeps_county_clean.csv', index=False)
print("OEPS data saved")

OEPS data saved


In [57]:
census = pd.read_csv('data/census_acs_dp03.csv', 
                     dtype={'GEO_ID': str},
                     skiprows=[1])

# All 4 correct columns
census = census[['GEO_ID', 'DP03_0009PE', 'DP03_0062E', 'DP03_0099PE', 'DP03_0128PE']]

census.columns = ['geo_id', 'unemployment_rate', 'median_income', 'uninsured_rate', 'poverty_rate']

census['county_fips'] = census['geo_id'].str[-5:]
census = census.drop(columns=['geo_id'])
census = census[census['county_fips'] != '000US']

for col in ['unemployment_rate', 'median_income', 'uninsured_rate', 'poverty_rate']:
    census[col] = pd.to_numeric(census[col], errors='coerce')

census = census.dropna(subset=['median_income'])

print(census.shape)
print(census.describe())

(3220, 5)
       unemployment_rate  median_income  uninsured_rate  poverty_rate
count        3220.000000    3220.000000     3220.000000   3220.000000
mean            4.922112   65046.649068        9.157484     14.986335
std             2.776188   18388.683350        4.967768      7.626237
min             0.000000   16170.000000        0.000000      1.700000
25%             3.300000   54113.250000        5.600000     10.100000
50%             4.500000   63161.500000        8.000000     13.400000
75%             5.900000   73216.250000       11.500000     17.800000
max            31.100000  178707.000000       44.300000     64.700000


/var/folders/69/5q2wfk010yqdww01frjhhm2c0000gn/T/ipykernel_47103/2698768189.py:1: DtypeWarning: Columns (0: DP03_0025E, 1: DP03_0025M, 2: DP03_0062E, 3: DP03_0062M, 4: DP03_0067E, 5: DP03_0067M, 6: DP03_0069E, 7: DP03_0069M, 8: DP03_0086E, 9: DP03_0086M, 10: DP03_0087E, 11: DP03_0087M, 12: DP03_0090E, 13: DP03_0090M, 14: DP03_0091E, 15: DP03_0091M, 16: DP03_0092E, 17: DP03_0092M, 18: DP03_0093E, 19: DP03_0093M, 20: DP03_0094E, 21: DP03_0094M, 22: DP03_0015PE, 23: DP03_0015PM, 24: DP03_0017PE, 25: DP03_0017PM, 26: DP03_0101PE, 27: DP03_0101PM, 28: DP03_0115PE, 29: DP03_0115PM, 30: DP03_0116PE, 31: DP03_0116PM, 32: DP03_0117PE, 33: DP03_0117PM, 34: DP03_0118PE, 35: DP03_0118PM, 36: DP03_0120PE, 37: DP03_0120PM, 38: DP03_0121PE, 39: DP03_0121PM, 40: DP03_0123PE, 41: DP03_0123PM, 42: DP03_0125PE, 43: DP03_0125PM, 44: DP03_0126PE, 45: DP03_0126PM, 46: DP03_0129PE, 47: DP03_0129PM, 48: DP03_0130PE, 49: DP03_0130PM, 50: DP03_0131PE, 51: DP03_0131PM, 52: DP03_0132PE, 53: DP03_0132PM) have mixe

In [58]:
print(census.describe())
print(census.isnull().sum())

       unemployment_rate  median_income  uninsured_rate  poverty_rate
count        3220.000000    3220.000000     3220.000000   3220.000000
mean            4.922112   65046.649068        9.157484     14.986335
std             2.776188   18388.683350        4.967768      7.626237
min             0.000000   16170.000000        0.000000      1.700000
25%             3.300000   54113.250000        5.600000     10.100000
50%             4.500000   63161.500000        8.000000     13.400000
75%             5.900000   73216.250000       11.500000     17.800000
max            31.100000  178707.000000       44.300000     64.700000
unemployment_rate    0
median_income        0
uninsured_rate       0
poverty_rate         0
county_fips          0
dtype: int64


In [59]:
census.to_csv('data/census_clean.csv', index=False)


In [60]:
cdc

,county_name,county_fips,year,deaths,population,crude_rate,is_suppressed
0,"Autauga County, AL",01001,2018,NaN,55601.0,NaN,True
1,"Autauga County, AL",01001,2019,NaN,55869.0,NaN,True
2,"Autauga County, AL",01001,2020,NaN,56145.0,NaN,True
3,"Autauga County, AL",01001,2021,10.0,59095.0,16.9,False
4,"Autauga County, AL",01001,2022,10.0,59759.0,16.7,False
...,...,...,...,...,...,...,...
18733,"Washakie County, WY",56043,2023,NaN,7710.0,NaN,True
18734,"Washakie County, WY",56043,2024,NaN,7662.0,NaN,True
18735,"Weston County, WY",56045,2018,NaN,6967.0,NaN,True
18736,"Weston County, WY",56045,2021,NaN,6745.0,NaN,True


In [61]:
cdc_agg = cdc.groupby('county_fips').agg(
    avg_deaths=('deaths', 'mean'),
    avg_crude_rate=('crude_rate', 'mean'),
    avg_population=('population', 'mean'),
    suppressed_years=('is_suppressed', 'sum'),
    total_years=('year', 'count')
).reset_index()

print(cdc_agg.shape)
print(cdc_agg.head())

(3059, 6)
  county_fips  avg_deaths  avg_crude_rate  avg_population  suppressed_years  \
0       01001   10.000000       16.733333    58325.000000                 4   
1       01003   61.000000       25.285714   238769.571429                 0   
2       01005   12.000000       48.600000    24681.285714                 6   
3       01007   11.000000       50.300000    22219.714286                 6   
4       01009   15.142857       25.657143    58868.142857                 0   

   total_years  
0            7  
1            7  
2            7  
3            7  
4            7  


In [62]:
# Merge CDC with OEPS pharmacy
master = cdc_agg.merge(county.rename(columns={'CountyGEOID': 'county_fips'}), 
                        on='county_fips', 
                        how='left')

# Merge with Census
master = master.merge(census, on='county_fips', how='left')

print(master.shape)
print(master.columns.tolist())
print(master.head())


(3059, 14)
['county_fips', 'avg_deaths', 'avg_crude_rate', 'avg_population', 'suppressed_years', 'total_years', 'TotTracts', 'PharCtTmDr', 'PharTmDrP', 'PharAvTmDr', 'unemployment_rate', 'median_income', 'uninsured_rate', 'poverty_rate']
  county_fips  avg_deaths  avg_crude_rate  avg_population  suppressed_years  \
0       01001   10.000000       16.733333    58325.000000                 4   
1       01003   61.000000       25.285714   238769.571429                 0   
2       01005   12.000000       48.600000    24681.285714                 6   
3       01007   11.000000       50.300000    22219.714286                 6   
4       01009   15.142857       25.657143    58868.142857                 0   

   total_years  TotTracts  PharCtTmDr  PharTmDrP  PharAvTmDr  \
0            7       17.0        17.0     100.00        9.91   
1            7       43.0        42.0      97.67        5.12   
2            7        9.0         8.0      88.89       14.71   
3            7        8.0      

In [63]:
print(master.isnull().sum())


county_fips             0
avg_deaths           1475
avg_crude_rate       1475
avg_population          0
suppressed_years        0
total_years             0
TotTracts               3
PharCtTmDr              3
PharTmDrP               3
PharAvTmDr             36
unemployment_rate      11
median_income          11
uninsured_rate         11
poverty_rate           11
dtype: int64


In [64]:
# Drop rows missing OEPS data (only 3)
master = master.dropna(subset=['TotTracts'])

# Drop rows missing Census data (only 11)
master = master.dropna(subset=['unemployment_rate'])

# Impute PharAvTmDr nulls with 120 minutes
# These counties had no calculable drive time in OSRM
# 120 mins represents extreme remoteness signa
master['PharAvTmDr'] = master['PharAvTmDr'].fillna(120)

print(master.shape)
print(master.isnull().sum())

(3048, 14)
county_fips             0
avg_deaths           1472
avg_crude_rate       1472
avg_population          0
suppressed_years        0
total_years             0
TotTracts               0
PharCtTmDr              0
PharTmDrP               0
PharAvTmDr              0
unemployment_rate       0
median_income           0
uninsured_rate          0
poverty_rate            0
dtype: int64


In [65]:
master.to_csv('data/master_county.csv', index=False)
print("Master table saved:", master.shape)

Master table saved: (3048, 14)
